**Séance :** 01  
**Nom :** FANOU Précieux  
**Porteur # :** P2  
**Étape assignée :** 1.2 --> 1.3 --> 1.5 --> 1.10

# Étape 1.10 : Le plancher

**Question** : si l'on n'a **aucun modèle**, quelle valeur unique annoncer pour tous les sinistres ? Et combien d'euros d'erreur cela coûte-t-il en moyenne ?

**Métrique** : la MAE (erreur absolue moyenne, en euros), car l'erreur se paie en euros.

| Constante testée | Ce qu'elle représente |
|---|---|
| la **moyenne** de `loss` | le montant moyen, tiré vers le haut par les gros sinistres |
| la **médiane** de `loss` | le montant du sinistre « du milieu » |

**Hypothèse à mesurer** : quand la métrique est la MAE, la meilleure constante est la **médiane**, pas la moyenne. L'énoncé donne le chiffre 1 807,09 (présenté comme la MAE de la moyenne) et un écart de 8,9 % entre moyenne et médiane. On **mesure** ces quantités et on les **compare** à l'énoncé, sans les supposer.

> Tout modèle construit ensuite devra battre **ce plancher**, pas zéro.

## Bloc 1 : imports et chemins des fichiers

In [1]:
# ── Ce que fait ce bloc : importer les outils et trouver où est train.csv ──
from pathlib import Path                              # chemins de fichiers
import numpy as np                                    # calcul sur les tableaux de nombres
import pandas as pd                                   # lecture du CSV et tableaux
from sklearn.model_selection import train_test_split  # pour le découpage de validation

# Sur Kaggle, les données sont dans /kaggle/input/... ; en local, dans le dossier courant
RACINE_KAGGLE = Path("/kaggle/input")

if RACINE_KAGGLE.exists():
    # On cherche train.csv UNIQUEMENT dans le jeu du sujet 04
    trouves = [p for p in RACINE_KAGGLE.rglob("train.csv") if "sujet-04" in str(p)]
    if not trouves:
        raise FileNotFoundError("train.csv du sujet 04 introuvable : ajoutez votre jeu (Add Input).")
    DOSSIER = trouves[0].parent
else:
    DOSSIER = Path(".")

CHEMIN_TRAIN = DOSSIER / "train.csv"

GRAINE = 42   # graine aléatoire fixée : le découpage sera identique à chaque exécution

print("Dossier des données :", DOSSIER)
print("train.csv existe    :", CHEMIN_TRAIN.exists())

# ── Résultat attendu ──
# Dossier des données : /kaggle/input/datasets/prcieuxfanou/esgc-vak-sujet-04-fanou   (en local : .)
# train.csv existe    : True

Dossier des données : .
train.csv existe    : True


## Bloc 2 : lire seulement `id` et `loss`

Le plancher n'utilise **aucune variable explicative** : il suffit de la colonne `loss`. On ne lit donc que `id` et `loss`, ce qui est plus rapide et plus léger.

In [2]:
# ── Ce que fait ce bloc : charger uniquement les colonnes utiles au plancher ──
train = pd.read_csv(CHEMIN_TRAIN, usecols=["id", "loss"])   # usecols : on ignore les 130 autres colonnes
train["loss"] = train["loss"].astype("float64")

loss = train["loss"]    # la cible : le montant du sinistre en euros

print("Forme :", train.shape)
print("Montant minimum :", round(float(loss.min()), 2))
print("Montant maximum :", round(float(loss.max()), 2))

# ── Résultat attendu ──
# Forme : (150654, 2)
# Montant minimum : 0.67
# Montant maximum : environ 121012

Forme : (150654, 2)
Montant minimum : 0.67
Montant maximum : 121012.25


## Bloc 3 : les deux constantes et leur MAE sur tout `train`

On calcule la moyenne et la médiane de `loss`, puis la MAE obtenue en annonçant toujours cette valeur. On comparera ces mesures au chiffre de l'énoncé (1 807,09) au bloc 8.

In [3]:
# ── Ce que fait ce bloc : MAE de la moyenne et de la médiane, calculées sur tout train ──
def mae(vraies, prediction):
    """Erreur absolue moyenne : de combien d'euros se trompe-t-on en moyenne ?"""
    return float(np.mean(np.abs(vraies - prediction)))

moyenne = float(loss.mean())      # la moyenne de loss
mediane = float(loss.median())    # la médiane de loss

mae_moyenne = mae(loss, moyenne)  # erreur si on annonce toujours la moyenne
mae_mediane = mae(loss, mediane)  # erreur si on annonce toujours la médiane

# Écart entre les deux MAE, avec DEUX définitions (il faut dire laquelle on utilise)
ecart_pct = (mae_moyenne - mae_mediane) / mae_moyenne * 100            # relatif à la MAE de la moyenne
ecart_pct_mediane = (mae_moyenne - mae_mediane) / mae_mediane * 100    # relatif à la MAE de la médiane

print(f"Moyenne de loss : {moyenne:.2f}")
print(f"Médiane de loss : {mediane:.2f}")
print()
print(f"MAE en annonçant la moyenne : {mae_moyenne:.2f}")
print(f"MAE en annonçant la médiane : {mae_mediane:.2f}")
print(f"Écart, relatif à la MAE de la moyenne : {ecart_pct:.1f} %")
print(f"Écart, relatif à la MAE de la médiane : {ecart_pct_mediane:.1f} %")

# ── Résultat attendu (mesuré sur le train.csv complet) ──
# Moyenne de loss : 3038.92 ; médiane de loss : 2115.57
# MAE en annonçant la moyenne : 1968.14
# MAE en annonçant la médiane : 1810.49
# Écart relatif à la MAE de la moyenne : 8.0 % ; relatif à celle de la médiane : 8.7 %
# La médiane est nettement inférieure à la moyenne : distribution très asymétrique.
# Ces valeurs sont comparées à celles de l'énoncé au bloc 8, sans être forcées.

Moyenne de loss : 3038.92
Médiane de loss : 2115.57

MAE en annonçant la moyenne : 1968.14
MAE en annonçant la médiane : 1810.49
Écart, relatif à la MAE de la moyenne : 8.0 %
Écart, relatif à la MAE de la médiane : 8.7 %


## Bloc 4 : pourquoi la médiane ? on le mesure

On teste plusieurs constantes (différents quantiles de `loss`) et on regarde laquelle donne la plus petite MAE. Si l'hypothèse est juste, le minimum se trouve au quantile 0,50, c'est-à-dire la médiane.

In [4]:
# ── Ce que fait ce bloc : chercher quelle constante minimise la MAE ──
quantiles = [0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70]

lignes = []
for q in quantiles:
    constante = float(loss.quantile(q))        # la valeur de loss au quantile q
    lignes.append({"quantile": q, "constante": round(constante, 2), "MAE": round(mae(loss, constante), 2)})

# On ajoute la moyenne pour la comparer
lignes.append({"quantile": "moyenne", "constante": round(moyenne, 2), "MAE": round(mae_moyenne, 2)})

essais = pd.DataFrame(lignes)
display(essais)

# Quel quantile donne la plus petite MAE ?
meilleur = essais[essais["quantile"] != "moyenne"].sort_values("MAE").iloc[0]
print("Quantile qui minimise la MAE :", meilleur["quantile"])

# ── Résultat attendu ──
# La MAE est la plus petite au quantile 0.5 (la médiane) et remonte de part et d'autre.
# La moyenne donne une MAE plus élevée.

,quantile,constante,MAE
0,0.3,1360.13,1952.46
1,0.4,1703.63,1850.21
2,0.45,1898.23,1821.21
3,0.5,2115.57,1810.49
4,0.55,2363.37,1823.18
5,0.6,2647.26,1866.11
6,0.7,3374.38,2088.73
7,moyenne,3038.92,1968.14


Quantile qui minimise la MAE : 0.5


## Bloc 5 : le découpage de validation (honnête)

Pour juger un modèle plus tard, on garde **une partie de `train` cachée**. Deux règles :

1. La constante est calculée sur la partie « apprentissage » **seulement**, jamais sur la partie cachée.
2. La cible est un **montant très asymétrique** : on **stratifie** sur des tranches de `loss` pour que les gros sinistres soient répartis dans les deux parties.

In [5]:
# ── Ce que fait ce bloc : couper train en 80 % apprentissage / 20 % validation, stratifié sur loss ──
# 1) On range chaque sinistre dans l'une de 10 tranches de montant (déciles)
tranches = pd.qcut(loss, q=10, labels=False, duplicates="drop")

# 2) On coupe en gardant la même répartition des tranches dans les deux parties
indices_app, indices_val = train_test_split(
    train.index, test_size=0.20, random_state=GRAINE, stratify=tranches
)

app = train.loc[indices_app]    # partie pour apprendre (ici : calculer les constantes)
val = train.loc[indices_val]    # partie cachée, pour mesurer honnêtement

print("Lignes d'apprentissage :", len(app))
print("Lignes de validation   :", len(val))
print("Total                  :", len(app) + len(val))

# Vérification : la queue est-elle bien répartie dans les deux parties ?
repere = pd.DataFrame({
    "apprentissage": app["loss"].describe(percentiles=[0.5, 0.99]),
    "validation":    val["loss"].describe(percentiles=[0.5, 0.99]),
}).round(2)
display(repere)

# ── Résultat attendu ──
# Lignes d'apprentissage : 120523 ; lignes de validation : 30131 ; total : 150654.
# Médiane, 99e percentile et maximum proches dans les deux parties.

Lignes d'apprentissage : 120523
Lignes de validation   : 30131
Total                  : 150654


,apprentissage,validation
count,120523.00,30131.00
mean,3038.83,3039.29
std,2928.68,2859.73
min,0.67,36.00
50%,2115.58,2115.56
99%,14001.15,14113.35
max,121012.25,37313.28


## Bloc 6 : le plancher honnête, mesuré sur la partie cachée

On apprend les constantes sur 80 % de `train` et on mesure la MAE sur les 20 % cachés. C'est **ce chiffre-là** que tout modèle devra battre.

In [6]:
# ── Ce que fait ce bloc : plancher mesuré sur la validation, constantes apprises sur l'apprentissage ──
moyenne_app = float(app["loss"].mean())      # constante apprise SANS regarder la validation
mediane_app = float(app["loss"].median())

mae_val_moyenne = mae(val["loss"], moyenne_app)
mae_val_mediane = mae(val["loss"], mediane_app)
ecart_val_pct = (mae_val_moyenne - mae_val_mediane) / mae_val_moyenne * 100

plancher = pd.DataFrame({
    "Constante annoncée": ["moyenne", "médiane"],
    "Valeur (apprise sur 80 %)": [round(moyenne_app, 2), round(mediane_app, 2)],
    "MAE sur validation (20 %)": [round(mae_val_moyenne, 2), round(mae_val_mediane, 2)],
})
display(plancher)

print(f"Écart moyenne -> médiane sur la validation : {ecart_val_pct:.1f} %")
print(f"PLANCHER À BATTRE (MAE de la médiane, validation) : {mae_val_mediane:.2f}")

# ── Résultat attendu ──
# La MAE de la médiane est inférieure à celle de la moyenne (mesuré : 1810.40 contre 1967.33).
# Le plancher à battre est la MAE de la médiane sur la validation.

,Constante annoncée,Valeur (apprise sur 80 %),MAE sur validation (20 %)
0,moyenne,3038.83,1967.33
1,médiane,2115.58,1810.40


Écart moyenne -> médiane sur la validation : 8.0 %
PLANCHER À BATTRE (MAE de la médiane, validation) : 1810.40


## Bloc 7 : où se trouve l'erreur ? sous-estimer ou surestimer, et la queue

Sous-estimer un sinistre (provision insuffisante) et le surestimer (argent immobilisé) ne coûtent pas la même chose pour l'assureur. On mesure ici comment l'erreur de la médiane se partage, et ce que pèsent les gros sinistres.

In [7]:
# ── Ce que fait ce bloc : décomposer l'erreur de la médiane (sur tout train) ──
ecart = loss - mediane            # positif : le vrai montant dépasse l'annonce (on a SOUS-estimé)

sous = ecart[ecart > 0]           # sinistres sous-estimés
sur = -ecart[ecart < 0]           # sinistres surestimés (on met l'écart en valeur positive)

print(f"Part des sinistres sous-estimés : {len(sous) / len(loss) * 100:.1f} %")
print(f"Part des sinistres surestimés   : {len(sur) / len(loss) * 100:.1f} %")
print(f"Euros de sous-estimation (somme)  : {sous.sum():,.0f}")
print(f"Euros de surestimation (somme)    : {sur.sum():,.0f}")

# La queue : que pèsent les plus gros sinistres dans l'erreur totale ?
erreur_abs = ecart.abs().sort_values(ascending=False)
total = erreur_abs.sum()

lignes = []
un_pour_cent = int(len(loss) * 0.01)   # 1 % des sinistres
for k in [10, 100, un_pour_cent]:      # les 10, 100 et 1 % de sinistres à plus forte erreur
    lignes.append({"sinistres les plus mal estimés": k,
                   "part de l'erreur totale (%)": round(erreur_abs.head(k).sum() / total * 100, 1)})
display(pd.DataFrame(lignes))

# Les 10 sinistres les plus lourds : montant réel contre annonce
lourds = train.nlargest(10, "loss")[["id", "loss"]].copy()
lourds["annonce_mediane"] = round(mediane, 2)
lourds["erreur"] = (lourds["loss"] - mediane).round(2)
display(lourds)

# ── Résultat attendu ──
# Environ la moitié des sinistres sont sous-estimés ; mais l'erreur en euros due aux sous-estimations est
# beaucoup plus grande que celle des surestimations : la queue (gros sinistres) décide de l'erreur.
# Quelques centaines de sinistres portent une part élevée de l'erreur totale.

Part des sinistres sous-estimés : 50.0 %
Part des sinistres surestimés   : 50.0 %
Euros de sous-estimation (somme)  : 205,931,880
Euros de surestimation (somme)    : 66,825,089


,sinistres les plus mal estimés,part de l'erreur totale (%)
0,10,0.3
1,100,1.2
2,1506,9.1


,id,loss,annonce_mediane,erreur
3130,167755.0,121012.25,2115.57,118896.68
99741,262556.0,106863.00,2115.57,104747.43
107179,47526.0,85923.56,2115.57,83807.99
6198,539471.0,79623.52,2115.57,77507.95
12022,254927.0,67667.16,2115.57,65551.59
67233,353451.0,59826.79,2115.57,57711.22
93973,495045.0,54190.39,2115.57,52074.82
97719,41419.0,51383.04,2115.57,49267.47
93526,370797.0,51127.50,2115.57,49011.93
105953,129560.0,49549.85,2115.57,47434.28


## Bloc 8 : tableau récapitulatif de l'étape 1.10

Deux tableaux : les **contrôles internes** du notebook (attendu = ce que doit donner un calcul correct), puis la **comparaison avec l'énoncé**, présentée comme une **information** : un écart y est à noter dans le journal, pas à corriger.

In [8]:
# ── Ce que fait ce bloc : contrôles internes, puis comparaison avec l'énoncé ──
quantile_minimal = float(meilleur["quantile"])                 # quantile qui minimise la MAE (bloc 4)
ids_communs = len(set(app["id"]) & set(val["id"]))             # un id dans les deux parties = fuite

def joli(v):
    """Affiche 150654 plutôt que 150654.00, et 0.5 tel quel."""
    return str(int(v)) if float(v).is_integer() else str(round(float(v), 2))

# 1) Contrôles internes : ce qu'un calcul correct doit donner
controles = [
    ("Lignes de train",                                   150654, len(train)),
    ("Lignes apprentissage + validation",                 150654, len(app) + len(val)),
    ("Part de validation (arrondie)",                     0.2,    round(len(val) / len(train), 2)),
    ("id communs apprentissage / validation",             0,      ids_communs),
    ("La médiane bat la moyenne, tout train (1 = oui)",   1,      int(mae_mediane < mae_moyenne)),
    ("Quantile qui minimise la MAE",                      0.5,    quantile_minimal),
    ("La médiane bat la moyenne, validation (1 = oui)",   1,      int(mae_val_mediane < mae_val_moyenne)),
]

recap = pd.DataFrame(controles, columns=["Contrôle", "Attendu", "Obtenu"])
recap["Verdict"] = recap.apply(
    lambda ligne: "OK" if ligne["Obtenu"] == ligne["Attendu"] else "ATTENTION", axis=1
)
recap["Attendu"] = recap["Attendu"].map(joli)   # mise en forme APRES la comparaison
recap["Obtenu"] = recap["Obtenu"].map(joli)
display(recap)

# 2) Comparaison avec l'énoncé : INFORMATION, pas de verdict forcé
REF_ENONCE = 1807.09        # chiffre de l'énoncé, présenté comme la MAE de la moyenne
ECART_ENONCE_PCT = 8.9      # écart moyenne / médiane annoncé par l'énoncé

comparaison = pd.DataFrame({
    "Quantité mesurée": ["MAE de la moyenne (tout train)", "MAE de la médiane (tout train)"],
    "Valeur mesurée": [round(mae_moyenne, 2), round(mae_mediane, 2)],
})
comparaison["Chiffre de l'énoncé"] = REF_ENONCE
comparaison["Écart (euros)"] = (comparaison["Valeur mesurée"] - REF_ENONCE).round(2)
comparaison["Écart relatif (%)"] = ((comparaison["Valeur mesurée"] / REF_ENONCE - 1) * 100).round(1)
display(comparaison)

plus_proche = comparaison.loc[comparaison["Écart (euros)"].abs().idxmin(), "Quantité mesurée"]
print("Quantité mesurée la plus proche de", REF_ENONCE, ":", plus_proche)
print(f"Écart moyenne -> médiane mesuré : {ecart_pct:.1f} % (relatif à la moyenne), "
      f"{ecart_pct_mediane:.1f} % (relatif à la médiane) ; énoncé : {ECART_ENONCE_PCT} %")

# 3) Le résultat à retenir
resume = pd.DataFrame({
    "Constante": ["moyenne", "médiane"],
    "Valeur annoncée (tout train)": [round(moyenne, 2), round(mediane, 2)],
    "MAE sur tout train": [round(mae_moyenne, 2), round(mae_mediane, 2)],
    "MAE sur validation (honnête)": [round(mae_val_moyenne, 2), round(mae_val_mediane, 2)],
})
display(resume)

print(f"Constante à annoncer sans modèle : {mediane:.2f} (médiane de loss sur tout train)")
print(f"Plancher à battre (validation)   : {mae_val_mediane:.2f}")

if (recap["Verdict"] == "OK").all():
    print("Étape 1.10 validée : le plancher est mesuré, les contrôles internes passent.")
else:
    print("ATTENTION : au moins un contrôle interne ne passe pas. Regarder la colonne Verdict.")

# ── Résultat attendu (mesuré sur le train.csv complet) ──
# Tableau de 7 contrôles, tous en OK.
# Comparaison : la MAE de la médiane (1810.49) est très proche du chiffre 1807.09 de l'énoncé
# (écart d'environ 0,2 %), alors que celle de la moyenne (1968.14) en est éloignée (environ 8,9 %).
# C'est une information à NOTER dans le journal (l'énoncé présente 1807.09 comme la MAE de la moyenne) :
# on ne modifie pas la mesure pour la faire coïncider. Question à poser à l'enseignant.
# Plancher à battre : 1810.40 (MAE de la médiane sur la validation).

,Contrôle,Attendu,Obtenu,Verdict
0,Lignes de train,150654,150654,OK
1,Lignes apprentissage + validation,150654,150654,OK
2,Part de validation (arrondie),0.2,0.2,OK
3,id communs apprentissage / validation,0,0,OK
4,"La médiane bat la moyenne, tout train (1 = oui)",1,1,OK
5,Quantile qui minimise la MAE,0.5,0.5,OK
6,"La médiane bat la moyenne, validation (1 = oui)",1,1,OK


,Quantité mesurée,Valeur mesurée,Chiffre de l'énoncé,Écart (euros),Écart relatif (%)
0,MAE de la moyenne (tout train),1968.14,1807.09,161.05,8.9
1,MAE de la médiane (tout train),1810.49,1807.09,3.40,0.2


Quantité mesurée la plus proche de 1807.09 : MAE de la médiane (tout train)
Écart moyenne -> médiane mesuré : 8.0 % (relatif à la moyenne), 8.7 % (relatif à la médiane) ; énoncé : 8.9 %


,Constante,Valeur annoncée (tout train),MAE sur tout train,MAE sur validation (honnête)
0,moyenne,3038.92,1968.14,1967.33
1,médiane,2115.57,1810.49,1810.40


Constante à annoncer sans modèle : 2115.57 (médiane de loss sur tout train)
Plancher à battre (validation)   : 1810.40
Étape 1.10 validée : le plancher est mesuré, les contrôles internes passent.
